# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rajhoou/MACHINE-LEARNING-NEW-PROJECT-/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### 1. My lane (or freestyle) and why

 Content Refresh Prioritization (Lane 1 — identifying which existing pages should be prioritized for a content refresh).

I chose this lane because my initial discovery run showed clear differences in `content_age_days` across `trend_direction` groups (`declining`, `stable`, `growing`) alongside a measurable relationship between `avg_position` and `ctr`. Prioritizing refreshes is a practical search-analytics problem where combining historical page metrics with structured decision criteria can directly help content teams focus their time on pages most likely to benefit, while I still have until the end of Week 4 to refine or switch lanes if needed.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

2. The question: decision, action, cost of a wrong call

Research Question: Among active search pages, which page-level patterns (such as content age, recent trend direction, average ranking position, and click-through rate) best distinguish pages showing sustained organic decline from pages that are stable or growing?
* Unit of Analysis: One individual page (`page_id` / row in the anonymized search dataset).
* Model / Analysis Output: A prioritized risk score or ranked shortlist flagging pages for editorial refresh review.
* Decision & Action:A content strategist or editor uses this ranked list during weekly planning to decide **which existing pages to audit and refresh first** versus which pages to leave untouched.
* Cost of a Wrong Call:
  * False Positive (flagging a healthy/stable page):Wastes editorial and SEO hours rewriting a page that did not need changes—and risks accidentally disrupting a page that was already performing well.
  * False Negative (missing a genuinely declining page):Lets a high-potential page continue losing organic clicks and visibility while editorial effort goes to lower-impact pages.
* Why This Is More Than "Just Train a Model": A raw classifier without clear thresholds, error-cost trade-offs, and actionable features produces scores an editor cannot trust or act on. Defining what counts as a meaningful decline and balancing editorial capacity against false-alarm rates matters just as much as the algorithm itself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

3. Quick look at the data (2–3 real numbers)

In the starter dataset (`30,000` active page rows across `44` columns), three observed patterns make this lane worth investigating over the next 7 weeks:
1. Scale (`30,000` active pages):The sample is large enough to compare segments across trend directions and age buckets without relying on tiny counts.
2. Position vs. CTR relationship (`Spearman r = -0.1444`, `Pearson r = -0.0726`): Higher (numerically lower) average rank positions correlate directionally with higher click-through rates, showing that rank shifts connect to measurable click differences, while leaving plenty of variance explained by other page factors.
3. Content age across `trend_direction`:Grouping `content_age_days` by `trend_direction` shows distinct age distributions between declining, stable, and growing pages, supporting the hypothesis that content freshness and historical performance signals carry useful signal for refresh triage.

In [2]:
import os
from pathlib import Path

repo_dir = Path("/content/flyrank-ml-internship-starter")
if not repo_dir.exists():
    # Replace with your own public repo URL if named differently
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git {repo_dir}

os.chdir(repo_dir)
print("Current working directory:", os.getcwd())

Cloning into '/content/flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 5.10 MiB/s, done.
Resolving deltas: 100% (161/161), done.
Current working directory: /content/flyrank-ml-internship-starter


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

# Locate the starter dataset whether running from repo root or work/notebooks/
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Dataset not found. Clone your repo in Colab first or update csv_path.")

df = pd.read_csv(csv_path)
print(f"Total active rows: {len(df):,} | Columns: {df.shape[1]}")

spearman_r = df["avg_position"].corr(df["ctr"], method="spearman")
pearson_r = df["avg_position"].corr(df["ctr"], method="pearson")
print(f"avg_position vs ctr -> Spearman: {spearman_r:.4f}, Pearson: {pearson_r:.4f}")

if "trend_direction" in df.columns and "content_age_days" in df.columns:
    print("\nContent age (days) by trend_direction:")
    print(df.groupby("trend_direction")["content_age_days"].agg(["count", "mean", "median"]).round(1))

Total active rows: 30,000 | Columns: 44
avg_position vs ctr -> Spearman: -0.1444, Pearson: -0.0726

Content age (days) by trend_direction:
                 count   mean  median
trend_direction                      
down             16262  236.2   216.0
flat              1152  245.9   231.0
new               2236  238.7   279.0
stable            5962  295.4   300.0
up                4388  288.5   291.5


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### 4. Careful words: what I can and can't claim

* **What I CAN claim:** The patterns reported here are **observed, directional associations** within this anonymized 30,000-page historical sample. They show how age, trend labels, position, and CTR co-vary in past FlyRank search records and can help prioritize pages for human editorial review.
* **What I CANNOT claim:** I cannot claim to have uncovered Google's search ranking algorithm, nor can I claim causal proof that refreshing a page with certain metrics will automatically boost its rankings or clicks. All identifiers remain anonymized, and no private client details are exposed or inferred.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.